In [45]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import joblib

In [46]:
df = pd.read_csv("mfcc_dataset.csv")

X = df.iloc[:, 1:].values.astype(np.float32)
y = df.iloc[:, 0].values

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))

Dataset shape: (12, 13)
Labels: ['a' 'b' 'c' 'd' 'e' 'kukur']


In [47]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

num_classes = len(np.unique(y_encoded))

print("Classes:", label_encoder.classes_)

Classes: ['a' 'b' 'c' 'd' 'e' 'kukur']


Normalize Features


In [48]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

Train/Test Split

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [50]:
class MFCCDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [51]:
train_dataset = MFCCDataset(X_train, y_train)
test_dataset = MFCCDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

Neural Network


In [52]:
class MFCCNet(nn.Module):

    def __init__(self, input_size=13, num_classes=3):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = MFCCNet(input_size=13, num_classes=num_classes)

In [53]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [54]:
epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")

Epoch 1/50 | Loss: 1.7821
Epoch 2/50 | Loss: 1.7582
Epoch 3/50 | Loss: 1.7347
Epoch 4/50 | Loss: 1.7117
Epoch 5/50 | Loss: 1.6891
Epoch 6/50 | Loss: 1.6671
Epoch 7/50 | Loss: 1.6456
Epoch 8/50 | Loss: 1.6243
Epoch 9/50 | Loss: 1.6031
Epoch 10/50 | Loss: 1.5825
Epoch 11/50 | Loss: 1.5625
Epoch 12/50 | Loss: 1.5425
Epoch 13/50 | Loss: 1.5227
Epoch 14/50 | Loss: 1.5032
Epoch 15/50 | Loss: 1.4835
Epoch 16/50 | Loss: 1.4634
Epoch 17/50 | Loss: 1.4431
Epoch 18/50 | Loss: 1.4224
Epoch 19/50 | Loss: 1.4015
Epoch 20/50 | Loss: 1.3805
Epoch 21/50 | Loss: 1.3597
Epoch 22/50 | Loss: 1.3389
Epoch 23/50 | Loss: 1.3177
Epoch 24/50 | Loss: 1.2962
Epoch 25/50 | Loss: 1.2742
Epoch 26/50 | Loss: 1.2518
Epoch 27/50 | Loss: 1.2290
Epoch 28/50 | Loss: 1.2058
Epoch 29/50 | Loss: 1.1826
Epoch 30/50 | Loss: 1.1591
Epoch 31/50 | Loss: 1.1356
Epoch 32/50 | Loss: 1.1120
Epoch 33/50 | Loss: 1.0881
Epoch 34/50 | Loss: 1.0639
Epoch 35/50 | Loss: 1.0394
Epoch 36/50 | Loss: 1.0148
Epoch 37/50 | Loss: 0.9902
Epoch 38/5

In [55]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

print("Accuracy:", correct / total)

Accuracy: 0.6666666666666666


In [56]:
torch.save(model.state_dict(), "mfcc_model.pth")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Model saved successfully.")

Model saved successfully.


In [57]:
import torch
import numpy as np
import joblib

import DCT  # your custom MFCC generator


# ==========================================================
# LOAD TRAINED MODEL + PREPROCESSORS
# ==========================================================

model = MFCCNet(input_size=13, num_classes=num_classes)
model.load_state_dict(torch.load("mfcc_model.pth"))
model.eval()

scaler = joblib.load("scaler.pkl")
label_encoder = joblib.load("label_encoder.pkl")

predict using dct.py mfcc

In [58]:
def predict_from_dct():

    # ======================================================
    # 1. Run your MFCC pipeline (DCT.py already handles audio)
    # ======================================================
    MFCCS = DCT.get_mfcc()   # shape: (frames, 13)
    
    

    # ======================================================
    # 2. Convert to feature vector (same as training)
    # ======================================================
    sample = MFCCS.mean(axis=0).astype(np.float32)
    print("Feature vector is:", sample)

    # ======================================================
    # 3. Normalize using TRAINED scaler
    # ======================================================
    sample = scaler.transform([sample]).astype(np.float32)

    # ======================================================
    # 4. Convert to tensor
    # ======================================================
    sample = torch.tensor(sample, dtype=torch.float32)

    # ======================================================
    # 5. Predict
    # ======================================================
    with torch.no_grad():
        output = model(sample)
        pred = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([pred])[0]

In [62]:
import importlib, DCT, FFT, melFilterBank; importlib.reload(DCT); importlib.reload(FFT); importlib.reload(melFilterBank)

Total Frames  47


<module 'melFilterBank' from '/home/sas/Desktop/Git_hub/mfcc-speech-recognition/melFilterBank.py'>

In [63]:
result = predict_from_dct()

print("Predicted Label:", result)
if(result == "kukur"):
    print("Tai Kukur!")

Feature vector is: [-43.283043    11.019834    -5.182051     0.14370005  -3.541066
   9.536091     6.057926     2.8391743    1.1955217    1.1879374
   5.871901     6.0234833    7.8806405 ]
Predicted Label: kukur
Tai Kukur!


In [61]:
print(list(model.parameters())[0][0][:5])

tensor([ 0.0836,  0.1905, -0.0612, -0.2579,  0.2018], grad_fn=<SliceBackward0>)
